# SuperEEG analysis

The objective is to measure the quality of the SuperEEG model. We perform a leave-one-out analysis and evaluate the quality of the reconstruction. We can measure the quality of the model by examining statistics such as average and histogrammed RMSE and metrics of individual traces.

The leave-one-out analysis looks like this:

1. load in all the data (resample to 250Hz ?)
2. cull electrodes which do not pass a kurtosis test
3. cull brains with fewer than two (three?) remaining electrodes
4. compile all remaining electrode locations
5. for each brain:
    1. compute full-brain collelation matrix K using all other patients' data (build models from brains using locs, then build one model from those models)
    2. for each electrode in this brain:
        1. extract Y_ska by removing this electrode
        2. compute Y_skb = (K_ba*inv(K_aa) * Y_ska.T).T
        3. compare Y_skb with observed trace
6. do something with the predicted traces. correlation

### Load in the data

let's start off with a small dataset - Miller's faceshouses-basic. after loading them, compile the electrode locations and make brain objects with those locations

In [1]:
import supereeg as se
import os
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt
import seaborn as sns
import multiprocessing as mp
%matplotlib inline

In [25]:
data_dir = '../brain-lab-data/miller-BIDS/faceshouses-basic'
data_name = 'faceshouses-basic_ieeg.mat'
locs_name = 'electrodes.mat'

# read the subject ids from the directory
IDS = [x[-2:] for x in os.listdir(data_dir) if x.startswith('sub-') and 'jm' not in x]

# read in the data
DATA = {f'{x}': loadmat(f"{data_dir}/sub-{x}/ieeg/sub-{x}_{data_name}") for x in IDS}

# read in the electrode locations
for x in IDS:
    DATA[x]['locs'] = loadmat(f"{data_dir}/sub-{x}/ieeg/sub-{x}_{locs_name}")['locs']

# compile all locations
R = np.unique(np.concatenate([DATA[x]['locs'] for x in IDS]),axis=0)

# make brain objects for each subject using all locations
BOS = {f'{x}': se.Brain(data = DATA[x]['data'], locs = DATA[x]['locs'], 
                sample_rate = DATA[x]['srate']) for x in IDS}

### Kurtosis threshold

trim out the electrodes that don't pass the test. cull the brains that don't have enough electrodes

the Miller data don't include kurtosis and i think they're already trimmed

In [3]:
# put something here when there's data that need it

In [4]:
R = np.unique(np.concatenate([DATA[x]['locs'] for x in IDS]),axis=0)

### For each brain

compute fbcm excluding subject brain

### For each electrode

estimate activity and compare

multiprocessing architecture:
1. pool1 of 8 for subjects
    1. pool2 of 2 for trodes
    2. save results to file as they are completed
    3. close/join pool2
2. close/join pool1

### Make an output folder

In [89]:
try:
    out_dir = f'{data_dir}/out'
    os.mkdir(out_dir)
except:
    print('out directory exists')

out directory exists


### multi-threaded

In [90]:
def f1(sub):
    # make a model using all the brains except the subject at all locations
    mo = se.Model([BOS[x] for x in IDS if x != sub])
    
    # get the correlation matrix
    K = mo.get_model()

    def f2(trode):
        # get subject electrode locations
        locs = DATA[sub]['locs']
        
        # get Y_ska by removing this electrode
        Y_ska = DATA[sub]['data'][:,np.unique(np.where(~(locs == trode))[0])]

        # get the indices of R where this patient's electrodes are
        aa_mask = np.array([True if x in locs and (x != trode).any() else False for x in R])
        ba_mask = np.array([True if (x == trode).all() else False for x in R])

        # get K_aa and K_ba using these masks
        K_aa = K[np.ix_(aa_mask,aa_mask)]
        K_ba = K[np.ix_(ba_mask,aa_mask)]
        
        # compute Y_skb
        Y_skb = ((K_ba@np.linalg.inv(K_aa))@(Y_ska.T)).T

        # save predicted trace
        np.savetxt(f'{out_dir}/sub-{sub}_electrode-{i}_recon.csv')

    # iterate over the electrodes in this subject's brain
    with mp.Pool(2) as p2:
        p2.map(f2, DATA[sub]['locs'])
        p2.close()
        p2.join()

# iterate over the subjects
with mp.Pool(8) as p1:
    p1.map(f1, IDS)
    p1.close()
    p1.join()

KeyboardInterrupt: 

### single-threaded

In [91]:
# iterate over the subjects
for sub in IDS:
    # make a model using all the brains except the subject at all locations
    mo = se.Model([BOS[x] for x in IDS if x != sub], locs=R)
    
    # get the correlation matrix
    K = mo.get_model()

    # get subject electrode locations
    locs = DATA[sub]['locs']

    # iterate over the electrodes in this subject's brain
    for i,trode in enumerate(locs):
        
        # get Y_ska by removing this electrode
        Y_ska = DATA[sub]['data'][:,np.unique(np.where(~(DATA[sub]['locs'] == trode))[0])]

        # get the indices of R where this patient's electrodes are
        aa_mask = np.array([True if x in locs and (x != trode).any() else False for x in R])
        ba_mask = np.array([True if (x == trode).all() else False for x in R])

        # get K_aa and K_ba using these masks
        K_aa = K[np.ix_(aa_mask,aa_mask)]
        K_ba = K[np.ix_(ba_mask,aa_mask)]
        
        # compute Y_skb
        Y_skb = ((K_ba@np.linalg.inv(K_aa))@(Y_ska.T)).T

        # save predicted trace
        np.savetxt(f'{out_dir}/sub-{sub}_electrode-{i}_recon.csv', Y_skb)

TypeError: savetxt() missing 1 required positional argument: 'X'

In [41]:
sub = 'aa'
trode = DATA['aa']['locs'][0]
DATA[sub]['data'][:,np.unique(np.where(~(DATA[sub]['locs'] == trode))[0])].shape

(271400, 45)